In [ ]:
!mkdir datasets

In [ ]:
!cp -r '/content/drive/MyDrive/datasets/' '/content/datasets/synthetic_dataset'

In [5]:
!yolo task=detect mode=train model=yolov8x.pt imgsz=640 data=coco_risiko.yaml epochs=25 batch=8 name=yolov8n_custom

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
100% 131M/131M [00:00<00:00, 211MB/s]
Ultralytics 8.3.97 🚀 Python-3.11.11 torch-2.6.0+cu124 CUDA:0 (Tesla T4, 15095MiB)
engine/trainer: task=detect, mode=train, model=yolov8x.pt, data=coco_risiko.yaml, epochs=25, time=None, patience=100, batch=8, imgsz=640, save=True, save_period=-1, cache=False, device=None, workers=8, project=None, name=yolov8n_custom, exist_ok=False, pretrained=True, optimizer=auto, verbose=True, seed=0, deterministic=True, single_cls=False, rect=False, cos_lr=False, close_mosaic=10, resume=False, amp=True, fraction=1.0, profile=False, freeze=None, multi_scale=False, overlap_mask=True, mask_ratio=4, dropout=0.0, val=True, split=val, save_json=False, save_hybrid

In [16]:
import torch
from ultralytics import YOLO
from torch.nn.utils import prune

# Carica il modello YOLOv8
model = YOLO("runs/detect/yolov8n_custom/weights/best.pt")


# Pruniamo solo la backbone (evitiamo la testa di predizione)
for name, module in model.model.named_modules():  # Usa `model.model.named_modules()`
    if "backbone" in name and isinstance(module, torch.nn.Conv2d):
        prune.l1_unstructured(module, name="weight", amount=0.3)  # 30% di pruning
        prune.remove(module, "weight")  # Rimuove il pruning dopo l'applicazione

# Salva il modello in modo corretto
model.save("pruned_selective.pt")




In [17]:
!yolo task=detect mode=train model=pruned_selective.pt imgsz=640 data=coco_risiko.yaml epochs=25 batch=8 name=yolov8n_custom

Ultralytics 8.3.97 🚀 Python-3.11.11 torch-2.6.0+cu124 CUDA:0 (Tesla T4, 15095MiB)
engine/trainer: task=detect, mode=train, model=pruned_selective.pt, data=coco_risiko.yaml, epochs=25, time=None, patience=100, batch=8, imgsz=640, save=True, save_period=-1, cache=False, device=None, workers=8, project=None, name=yolov8n_custom3, exist_ok=False, pretrained=True, optimizer=auto, verbose=True, seed=0, deterministic=True, single_cls=False, rect=False, cos_lr=False, close_mosaic=10, resume=False, amp=True, fraction=1.0, profile=False, freeze=None, multi_scale=False, overlap_mask=True, mask_ratio=4, dropout=0.0, val=True, split=val, save_json=False, save_hybrid=False, conf=None, iou=0.7, max_det=300, half=False, dnn=False, plots=True, source=None, vid_stride=1, stream_buffer=False, visualize=False, augment=False, agnostic_nms=False, classes=None, retina_masks=False, embed=None, show=False, save_frames=False, save_txt=False, save_conf=False, save_crop=False, show_labels=True, show_conf=True, sh

In [ ]:
!yolo task=detect mode=predict model=pruned_selective.pt source=000853.jpg show=True imgsz=640 name=test conf=0.3

WARNING ⚠️ Environment does not support cv2.imshow() or PIL Image.show()

Ultralytics 8.3.97 🚀 Python-3.11.11 torch-2.6.0+cu124 CUDA:0 (Tesla T4, 15095MiB)
Model summary (fused): 112 layers, 68,135,124 parameters, 0 gradients, 257.4 GFLOPs

image 1/1 /content/000853.jpg: 448x640 3 blue_armys, 13 red_armys, 7 yellow_armys, 8 purple_armys, 6 black_armys, 5 green_armys, 1 blue_flag, 1 black_flag, 73.5ms
Speed: 2.8ms preprocess, 73.5ms inference, 181.4ms postprocess per image at shape (1, 3, 448, 640)
Results saved to runs/detect/test
💡 Learn more at https://docs.ultralytics.com/modes/predict
